# Libraries

In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Normalization, Dense, SimpleRNN, LSTM, GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Load data

In [2]:
# Read in the weather data csv
df = pd.read_csv('data/weather.csv', parse_dates=['id'])
df.head()

,id,split,MinTemp,MaxTemp,MedTemp,Rainfall,Sunshine,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Temp9am,Temp3pm
0,2008-02-01,labeled,19.5,22.4,20.95,15.6,0.0,92.0,84.0,1017.6,1017.4,20.7,20.9
1,2008-02-02,labeled,19.5,25.6,22.55,6.0,2.7,83.0,73.0,1017.9,1016.4,22.4,24.8
2,2008-02-03,labeled,21.6,24.5,23.05,6.6,0.1,88.0,86.0,1016.7,1015.6,23.5,23.0
3,2008-02-04,labeled,20.2,22.8,21.50,18.8,0.0,83.0,90.0,1014.2,1011.8,21.4,20.9
4,2008-02-05,labeled,19.7,25.7,22.70,77.4,0.0,88.0,74.0,1008.3,1004.8,22.5,25.5


In [3]:
df_lb = df[df['split'] == 'leaderboard'].copy()
df = df[df['split'] == 'labeled'].copy()

In [4]:
# For the example's simplicity, keep necessary columns only
df = df[['id', 'MedTemp']]
df

,id,MedTemp
0,2008-02-01,20.95
1,2008-02-02,22.55
2,2008-02-03,23.05
3,2008-02-04,21.50
4,2008-02-05,22.70
...,...,...
3324,2017-06-06,13.50
3325,2017-06-07,12.15
3326,2017-06-08,15.25
3327,2017-06-09,14.65


In [5]:
df['id'].diff().sort_values()

1       1 days
2214    1 days
2215    1 days
2216    1 days
2217    1 days
         ...  
3328    1 days
1766   29 days
1155   31 days
1735   32 days
0          NaT
Name: id, Length: 3329, dtype: timedelta64[ns]

## Date and time features

Add date and time features (columns) such as year, month, day, season, day of the week, etc.

Example:

In [6]:
df['Day_of_year'] = df['id'].dt.dayofyear
df['Month'] = df['id'].dt.month
df['Season'] = df['id'].dt.quarter
df.head()

,id,MedTemp,Day_of_year,Month,Season
0,2008-02-01,20.95,32,2,1
1,2008-02-02,22.55,33,2,1
2,2008-02-03,23.05,34,2,1
3,2008-02-04,21.50,35,2,1
4,2008-02-05,22.70,36,2,1


## Target

Compute the target value we want to predict with a prediction window of 1 day.

Feel free to try other prediction windows, e.g. forecasting 2, 3 or 7 days. Further predictions should have worse results, as the current temperature values are less informative of the far future.

In [7]:
df['target'] = df['MedTemp'].shift(-1).values
df

,id,MedTemp,Day_of_year,Month,Season,target
0,2008-02-01,20.95,32,2,1,22.55
1,2008-02-02,22.55,33,2,1,23.05
2,2008-02-03,23.05,34,2,1,21.50
3,2008-02-04,21.50,35,2,1,22.70
4,2008-02-05,22.70,36,2,1,23.70
...,...,...,...,...,...,...
3324,2017-06-06,13.50,157,6,2,12.15
3325,2017-06-07,12.15,158,6,2,15.25
3326,2017-06-08,15.25,159,6,2,14.65
3327,2017-06-09,14.65,160,6,2,15.30


## Clean NaNs

Drop the NaNs that have appeared during the feature engineering process.

In [8]:
df = df.dropna()
df.shape

(3328, 6)

# Prepare train / test

In [9]:
def create_data_for_rnns(X, y, sequence_size=7):
    dataX, dataY = [], []
    for i in range(len(X) - sequence_size):
        # Take sequence of length `sequence_size`
        seqX = X.iloc[i:(i + sequence_size)].values
        # seqY = y.iloc[i:(i + sequence_size)].values
        seqY = y.iloc[i + sequence_size]
        dataX.append(seqX)
        dataY.append(seqY)
    return np.array(dataX), np.array(dataY)

In [10]:
X = df.drop(['id', 'target'], axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [11]:
X_train.head()

,MedTemp,Day_of_year,Month,Season
0,20.95,32,2,1
1,22.55,33,2,1
2,23.05,34,2,1
3,21.50,35,2,1
4,22.70,36,2,1


In [12]:
# Example sequence size of 5
X_train_rnn, y_train_rnn = create_data_for_rnns(X_train, y_train, sequence_size=5)
X_test_rnn, y_test_rnn = create_data_for_rnns(X_test, y_test, sequence_size=5)

# Sample of first sequence
pd.DataFrame(X_train_rnn[0], columns=X_train.columns)

,MedTemp,Day_of_year,Month,Season
0,20.95,32.0,2.0,1.0
1,22.55,33.0,2.0,1.0
2,23.05,34.0,2.0,1.0
3,21.50,35.0,2.0,1.0
4,22.70,36.0,2.0,1.0


In [13]:
X_train_rnn.shape

(2657, 5, 4)

In [14]:
y_train_rnn.shape

(2657,)

In [15]:
# Take care, this creates a bit of leakage during the CV with training data!

normalization = Normalization()
normalization.adapt(X_train_rnn)

2025-12-03 09:21:37.268134: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Time Series Cross-Validation

In [16]:
# We'll move training data, only using the past ~8 years

tscv = TimeSeriesSplit(n_splits=10, test_size=7, max_train_size=365 * 8)

In [17]:
# Adapt sklearn's TimeSeriesSplit for keras models
def custom_tscv(tscv, X, y, model, fit_params={}):
    all_fold_metrics = []
    # Loop through each split
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        print(f'Fold {fold + 1} => ', end='')
        
        # Split into train/val sets
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        
        # Train the model (if it is a NN, update it starting from the last weights)
        model.fit(X_train, y_train, validation_data=(X_val, y_val), **fit_params)
        
        # Evaluate on val set
        
        if len(y.shape) == 1:
            val_loss = model.evaluate(X_val, y_val, verbose=0)
            all_fold_metrics.append(val_loss)
            print(f'Val Loss: {val_loss:.2f}')
        else:
            pred = model.predict(X_val, verbose=0)
            metrics = [mean_absolute_error(y_val[:, i], pred[:, i])
                       for i in range(pred.shape[1])]
            all_fold_metrics.append(metrics)
            # Print results
            print()
            for i, m in enumerate(metrics, start=1):
                print(f'    Horizon {i}: MAE = {m:.2f}')
            print()

    return all_fold_metrics

# Models

Here I just use simple RNNs, feel free to implement more complex networks, try different sequence sizes, etc.

## Simple

In [18]:
rnn = Sequential([
    Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    normalization,
    SimpleRNN(16, activation='tanh', return_sequences=True),
    SimpleRNN(16, activation='tanh', return_sequences=False),
    Dense(8, activation='relu'),
    Dense(1, activation='linear')
])

rnn.compile(optimizer=Adam(learning_rate=0.001), loss='mae')

In [19]:
es = EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True)
fit_params = {
    'epochs': 100,
    'batch_size': 64,
    'callbacks': [es],
    'verbose': 0,
}
val_losses = custom_tscv(tscv, X_train_rnn, y_train_rnn, rnn, fit_params)
print(f'\nMean Validation MAE: {round(np.mean(val_losses), 1)}')

Fold 1 => Val Loss: 1.44
Fold 2 => Val Loss: 2.72
Fold 3 => Val Loss: 4.43
Fold 4 => Val Loss: 1.01
Fold 5 => Val Loss: 0.49
Fold 6 => Val Loss: 1.36
Fold 7 => Val Loss: 0.97
Fold 8 => Val Loss: 1.66
Fold 9 => Val Loss: 2.56
Fold 10 => Val Loss: 1.27

Mean Validation MAE: 1.8


## LSTM

Looks like it is not working amazingly well... improve it :)

In [20]:
lstm = Sequential([
    Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    normalization,
    LSTM(16, return_sequences=True),
    LSTM(16, return_sequences=False),
    Dense(8, activation='relu'),
    Dense(1, activation='linear')
])

lstm.compile(optimizer=Adam(learning_rate=0.001), loss='mae')

In [21]:
es = EarlyStopping(patience=10, monitor='val_loss', restore_best_weights=True)
fit_params = {
    'epochs': 100,
    'batch_size': 64,
    'callbacks': [es],
    'verbose': 0,
}
val_losses = custom_tscv(tscv, X_train_rnn, y_train_rnn, lstm, fit_params)
print(f'\nMean Validation MAE: {round(np.mean(val_losses), 1)}')

Fold 1 => Val Loss: 1.18
Fold 2 => Val Loss: 2.45
Fold 3 => Val Loss: 3.92
Fold 4 => Val Loss: 5.10
Fold 5 => Val Loss: 6.80
Fold 6 => Val Loss: 7.07
Fold 7 => Val Loss: 5.41
Fold 8 => Val Loss: 4.66
Fold 9 => Val Loss: 5.42
Fold 10 => Val Loss: 4.93

Mean Validation MAE: 4.7


## GRU

Looks like it is not working amazingly well... improve it :)

In [22]:
gru = Sequential([
    Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    normalization,
    GRU(16, return_sequences=True),
    GRU(16, return_sequences=False),
    Dense(8, activation='relu'),
    Dense(1, activation='linear')
])

gru.compile(optimizer=Adam(learning_rate=0.001), loss='mae')

In [23]:
es = EarlyStopping(patience=10, monitor='val_loss', restore_best_weights=True)
fit_params = {
    'epochs': 200,
    'batch_size': 64,
    'callbacks': [es],
    'verbose': 0,
}
val_losses = custom_tscv(tscv, X_train_rnn, y_train_rnn, gru, fit_params)
print(f'\nMean Validation MAE: {round(np.mean(val_losses), 1)}')

Fold 1 => Val Loss: 1.27
Fold 2 => Val Loss: 2.46
Fold 3 => Val Loss: 3.94
Fold 4 => Val Loss: 5.15
Fold 5 => Val Loss: 6.84
Fold 6 => Val Loss: 6.91
Fold 7 => Val Loss: 5.30
Fold 8 => Val Loss: 4.77
Fold 9 => Val Loss: 5.38
Fold 10 => Val Loss: 4.91

Mean Validation MAE: 4.7


## Multi-day output example

One option could be having multiple models, each one trained with a specific target (predicting in 1 day, 2 days, etc.).

Another option is to directly predict N days in advance with N output neurons. For example, up to 3 days in advance with 3 output neurons, one for each prediction.

### Compute multi-target

In [24]:
def build_y(temp, horizon=3):
    temp = np.asarray(temp)
    N = len(temp)

    # Number of valid starting positions
    M = N - horizon

    y = np.zeros((M, horizon))

    for i in range(M):
        y[i] = temp[i : i+horizon]

    return y

In [25]:
y_train_rnn.shape

(2657,)

In [26]:
y_train_rnn[:6]

array([22.45, 19.75, 18.6 , 19.4 , 20.15, 23.1 ])

In [27]:
horizon = 3

X_train_rnn_multi = X_train_rnn[:-horizon]  # take out the last rows based on the horizon
y_train_rnn_multi = build_y(y_train_rnn, horizon)
y_train_rnn_multi

array([[22.45, 19.75, 18.6 ],
       [19.75, 18.6 , 19.4 ],
       [18.6 , 19.4 , 20.15],
       ...,
       [11.9 , 14.6 , 15.4 ],
       [14.6 , 15.4 , 12.95],
       [15.4 , 12.95, 14.6 ]])

In [28]:
X_train_rnn_multi.shape, y_train_rnn_multi.shape

((2654, 5, 4), (2654, 3))

In [29]:
rnn = Sequential([
    Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    normalization,
    SimpleRNN(16, activation='tanh', return_sequences=True),
    SimpleRNN(16, activation='tanh', return_sequences=False),
    Dense(8, activation='relu'),
    Dense(3, activation='linear')  # Output of 3 neurons
])

rnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mae'
)

In [30]:
es = EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True)
fit_params = {
    'epochs': 100,
    'batch_size': 64,
    'callbacks': [es],
    'verbose': 0,
}
val_scores = custom_tscv(tscv, X_train_rnn_multi, y_train_rnn_multi, rnn, fit_params)

# Mean validation score of all predictions together
print(f'\nMean Validation MAE: {round(np.mean(val_scores), 1)}')

Fold 1 => 
    Horizon 1: MAE = 2.13
    Horizon 2: MAE = 2.01
    Horizon 3: MAE = 2.18

Fold 2 => 
    Horizon 1: MAE = 0.97
    Horizon 2: MAE = 0.84
    Horizon 3: MAE = 0.84

Fold 3 => 
    Horizon 1: MAE = 1.72
    Horizon 2: MAE = 1.69
    Horizon 3: MAE = 2.06

Fold 4 => 
    Horizon 1: MAE = 0.80
    Horizon 2: MAE = 0.65
    Horizon 3: MAE = 0.64

Fold 5 => 
    Horizon 1: MAE = 2.04
    Horizon 2: MAE = 2.26
    Horizon 3: MAE = 2.16

Fold 6 => 
    Horizon 1: MAE = 0.60
    Horizon 2: MAE = 0.61
    Horizon 3: MAE = 0.91

Fold 7 => 
    Horizon 1: MAE = 1.57
    Horizon 2: MAE = 1.43
    Horizon 3: MAE = 1.46

Fold 8 => 
    Horizon 1: MAE = 1.66
    Horizon 2: MAE = 1.79
    Horizon 3: MAE = 1.65

Fold 9 => 
    Horizon 1: MAE = 1.83
    Horizon 2: MAE = 2.54
    Horizon 3: MAE = 2.86

Fold 10 => 
    Horizon 1: MAE = 2.17
    Horizon 2: MAE = 1.87
    Horizon 3: MAE = 1.43


Mean Validation MAE: 1.6


Note this RNN by default predicts all 1, 2 and 3 days in advance. For example, given the last 5 days of test as a sequence, predict the next three days temperatures. You can do something similar for the leaderboard.

In [31]:
X_test_rnn[-1:]

array([[[ 15.1 , 155.  ,   6.  ,   2.  ],
        [ 14.5 , 156.  ,   6.  ,   2.  ],
        [ 13.5 , 157.  ,   6.  ,   2.  ],
        [ 12.15, 158.  ,   6.  ,   2.  ],
        [ 15.25, 159.  ,   6.  ,   2.  ]]])

In [32]:
rnn.predict(X_test_rnn[-1:])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


array([[14.596861, 14.449884, 14.49478 ]], dtype=float32)

In [33]:
# What values should really be

y_test_rnn[-3:]

array([15.25, 14.65, 15.3 ])